# Liquidity Monitoring — Intraday, Funding Gap, Collateral, Stress, EWI and ILAAP

Operational and management-layer liquidity tools (BCBS 248, EBA/GL/2018/02,
EBA/GL/2021/01) that sit beneath the regulatory ratios and feed the ILAAP.

<small>

| Module | Tool | Reference |
|---|---|---|
| Intraday | Daily max usage, utilisation, peak hour | BCBS 248 |
| Funding gap | Maturity ladder, rollover risk | EBA ITS 2021/05 |
| Collateral | Encumbrance ratio, HQLA buffer | EBA ITS 2021/05 |
| Stress | Idiosyncratic / market-wide LCR | EBA/GL/2018/02 |
| EWI | Traffic-light dashboard | EBA/GL/2021/01 |
| ILAAP | Aggregated adequacy report | EBA/GL/2021/01 |

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from banking_risk.liquidity import (
    # LCR (needed for stress and ILAAP)
    HQLA_Asset, HQLA_Level, Cash_Outflow, Cash_Inflow,
    Outflow_Type, Inflow_Type, SA_LCR_Calculator,
    # NSFR (needed for ILAAP)
    ASF_Item, ASF_Category, RSF_Item, RSF_Category, SA_NSFR_Calculator,
    # Intraday
    Intraday_Payment, Intraday_Monitor,
    # Funding gap
    Funding_Item, Funding_Gap_Analyser, BUCKET_LABELS,
    # Collateral
    Collateral_Asset, Asset_Class, Collateral_Manager,
    # Stress
    Liquidity_Stress_Calculator,
    IDIOSYNCRATIC_SCENARIO, MARKET_WIDE_SCENARIO, COMBINED_SCENARIO,
    # EWI
    EWI_Monitor, lcr_indicator, nsfr_indicator,
    encumbrance_indicator, rollover_indicator, intraday_utilisation_indicator,
    # ILAAP
    ILAAP_Aggregator,
)
from banking_risk.utils.reporting import Dark_Style

Dark_Style().apply()
p = Dark_Style().palette

## 2. Intraday liquidity monitoring (BCBS 248)

The bank starts the day with EUR 500M of available intraday liquidity
(central bank account + intraday credit lines). Payments flow throughout
the day — RTGS settlement batches, CLS FX payments, client transactions.

In [ ]:
payments = [
    Intraday_Payment("CLS FX settlement",      8.0,  -180_000_000, "CLS"),
    Intraday_Payment("RTGS batch 1",           9.0,  -120_000_000, "TARGET2"),
    Intraday_Payment("Incoming RTGS",          9.5,    90_000_000, "TARGET2"),
    Intraday_Payment("Client payment out",    10.0,   -60_000_000, "Corp_A"),
    Intraday_Payment("Maturing repo inflow",  11.0,   150_000_000, "ECB"),
    Intraday_Payment("RTGS batch 2",          13.0,   -80_000_000, "TARGET2"),
    Intraday_Payment("Client payment in",     14.5,    40_000_000, "Corp_B"),
    Intraday_Payment("End-of-day settle",     16.0,   -30_000_000, "TARGET2"),
    Intraday_Payment("Overnight repo",        17.0,   100_000_000, "ECB"),
]

monitor = Intraday_Monitor(opening_balance=500_000_000)
intra   = monitor.compute(payments)

print(f"Opening balance    : EUR {intra.opening_balance/1e6:>7.0f}M")
print(f"Closing balance    : EUR {intra.closing_balance/1e6:>7.0f}M")
print(f"Minimum balance    : EUR {intra.minimum_balance/1e6:>7.0f}M  (at {intra.peak_hour:.1f}h)")
print(f"Maximum usage      : EUR {intra.maximum_usage/1e6:>7.0f}M")
print(f"Utilisation rate   : {intra.utilisation_rate:.1%}")
print(f"Payments sent      : EUR {intra.total_payments_sent/1e6:>7.0f}M")
print(f"Payments received  : EUR {intra.total_payments_received/1e6:>7.0f}M")

In [ ]:
ts = intra.time_series.copy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts["hour"], ts["running_balance"]/1e6, color=p["cyan"], lw=2, marker="o", ms=5)
ax.axhline(intra.opening_balance/1e6, color=p["text_muted"], lw=0.8, ls="--",
           label=f"Opening {intra.opening_balance/1e6:.0f}M")
ax.axhline(intra.minimum_balance/1e6, color=p["red"], lw=1.2, ls=":",
           label=f"Min balance {intra.minimum_balance/1e6:.0f}M (usage {intra.utilisation_rate:.0%})")
ax.fill_between(ts["hour"], ts["running_balance"]/1e6, intra.opening_balance/1e6,
                alpha=0.15, color=p["red"])
ax.set_xlabel("Hour of day")
ax.set_ylabel("Intraday balance (EUR M)")
ax.set_title("Intraday liquidity position — BCBS 248 monitoring", color=p["text_title"])
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Funding gap analysis

Contractual cash flows bucketed by remaining maturity. A negative cumulative gap
(more liabilities maturing than assets) signals rollover risk — the bank must
refinance or liquidate assets to meet maturing obligations.

In [ ]:
funding_items = [
    # Assets — cash coming back in
    Funding_Item("Overnight repo",         200_000_000,   1, "asset",     "ECB"),
    Funding_Item("T-bill 1W",              100_000_000,   5, "asset"),
    Funding_Item("T-bill 1M",              150_000_000,  28, "asset"),
    Funding_Item("Corp loan 3M",            80_000_000,  75, "asset",     "Corp_A"),
    Funding_Item("Residential mortgage 5Y",500_000_000,1825, "asset"),
    Funding_Item("Corp bond held 7Y",      200_000_000,2555, "asset"),
    # Liabilities — funding rolling off
    Funding_Item("Overnight repo (borrow)",180_000_000,   1, "liability", "BankA"),
    Funding_Item("1W interbank",           120_000_000,   5, "liability", "BankB"),
    Funding_Item("1M CD",                   90_000_000,  25, "liability", "BankC"),
    Funding_Item("3M CD",                  150_000_000,  80, "liability", "BankA"),
    Funding_Item("6M senior bond",         200_000_000, 170, "liability"),
    Funding_Item("2Y retail term deposit", 300_000_000, 720, "liability"),
    Funding_Item("5Y covered bond",        400_000_000,1800, "liability"),
]

gap_result = Funding_Gap_Analyser().compute(funding_items)

print(f"Rollover within 30 days  : EUR {gap_result.rollover_30d/1e6:>8.1f}M")
print(f"Rollover within 90 days  : EUR {gap_result.rollover_90d/1e6:>8.1f}M")
print(f"Rollover within 1 year   : EUR {gap_result.rollover_1y/1e6:>8.1f}M")
print(f"30-day rollover ratio     : {gap_result.rollover_ratio_30d:.1%}")

gap_result.gap_table.div(1e6).round(1)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

gt      = gap_result.gap_table / 1e6
buckets = gt.index.tolist()
x       = np.arange(len(buckets))
w       = 0.35

ax1.bar(x - w/2, gt["assets"],      w, color=p["cyan"],  alpha=0.85, label="Assets")
ax1.bar(x + w/2, gt["liabilities"], w, color=p["red"],   alpha=0.75, label="Liabilities")
ax1.set_ylabel("EUR millions")
ax1.set_title("Cash flows by maturity bucket", color=p["text_title"])
ax1.legend(fontsize=8)
ax1.grid(axis="y", alpha=0.3)

cum = gt["cumulative_gap"]
bar_colors = [p["cyan"] if v >= 0 else p["red"] for v in cum]
ax2.bar(x, cum, color=bar_colors, alpha=0.85)
ax2.axhline(0, color=p["text_muted"], lw=0.8, ls="--")
ax2.set_xticks(x)
ax2.set_xticklabels(buckets, rotation=30, ha="right", fontsize=8)
ax2.set_ylabel("EUR millions")
ax2.set_title("Cumulative funding gap (positive = surplus)", color=p["text_title"])
ax2.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

## 4. Collateral encumbrance

Distinguishes pledged (encumbered) from free assets and identifies the
unencumbered HQLA pool available as the contingency funding buffer.

In [ ]:
collateral_assets = [
    Collateral_Asset("ECB repo pool — Govts",    Asset_Class.GOVT_BOND,    400_000_000, encumbered=True,  haircut=0.02),
    Collateral_Asset("Covered bond pool",        Asset_Class.COVERED_BOND, 200_000_000, encumbered=True,  haircut=0.04),
    Collateral_Asset("Free govt bonds",          Asset_Class.GOVT_BOND,    300_000_000, encumbered=False, haircut=0.02),
    Collateral_Asset("Free covered bonds",       Asset_Class.COVERED_BOND, 150_000_000, encumbered=False, haircut=0.04),
    Collateral_Asset("IG corp bonds (free)",     Asset_Class.CORPORATE_BOND, 80_000_000, encumbered=False, haircut=0.10),
    Collateral_Asset("RMBS (free)",              Asset_Class.RMBS,          60_000_000, encumbered=False, haircut=0.08),
    Collateral_Asset("Loan book",                Asset_Class.LOAN,         800_000_000, encumbered=False, haircut=0.30),
    Collateral_Asset("Derivative margin posted", Asset_Class.OTHER,         50_000_000, encumbered=True,  haircut=0.0),
]

enc = Collateral_Manager().analyse(collateral_assets)

print(f"Total assets           : EUR {enc.total_assets/1e6:>7.0f}M")
print(f"Encumbered             : EUR {enc.encumbered/1e6:>7.0f}M  ({enc.encumbrance_ratio:.1%})")
print(f"Unencumbered           : EUR {enc.unencumbered/1e6:>7.0f}M")
print(f"Available HQLA buffer  : EUR {enc.available_hqla/1e6:>7.0f}M")
print(f"  Level 1              : EUR {enc.available_hqla_by_level['1']/1e6:>7.0f}M")
print(f"  Level 2A             : EUR {enc.available_hqla_by_level['2A']/1e6:>7.0f}M")
print(f"  Level 2B             : EUR {enc.available_hqla_by_level['2B']/1e6:>7.0f}M")
print(f"Available for repo     : EUR {enc.available_for_repo/1e6:>7.0f}M  (incl. non-HQLA)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Encumbered vs free
sizes  = [enc.encumbered/1e6, enc.unencumbered/1e6]
labels = [f"Encumbered\n{enc.encumbrance_ratio:.0%}", f"Unencumbered\n{1-enc.encumbrance_ratio:.0%}"]
ax1.pie(sizes, labels=labels, colors=[p["red"], p["cyan"]],
        autopct="%1.0f%%", startangle=90, textprops={"fontsize": 9})
ax1.set_title("Asset encumbrance", color=p["text_title"])

# Panel 2: HQLA buffer composition
hqla_labels = ["Level 1", "Level 2A", "Level 2B"]
hqla_vals   = [enc.available_hqla_by_level[k]/1e6 for k in ["1", "2A", "2B"]]
bar_c       = [p["cyan"], p["green"], p["amber"]]
ax2.bar(hqla_labels, hqla_vals, color=bar_c, alpha=0.85, width=0.5)
for i, v in enumerate(hqla_vals):
    ax2.text(i, v + 1, f"€{v:.0f}M", ha="center", fontsize=9, color=p["text_body"])
ax2.set_ylabel("EUR millions")
ax2.set_title("Unencumbered HQLA buffer (post-haircut)", color=p["text_title"])
ax2.grid(axis="y", alpha=0.3)

fig.suptitle("Collateral Encumbrance — EBA ITS 2021/05",
             color=p["text_title"], fontweight="bold")
fig.tight_layout()
plt.show()

## 5. Liquidity stress testing

Three standard EBA/GL/2018/02 scenarios applied on top of the base LCR inputs.
Each scenario multiplies outflow rates and adds haircuts to HQLA.

| Scenario | HQLA haircut add-on | Outflow multiplier | Inflow multiplier |
|---|---|---|---|
| Idiosyncratic | 0 pp | ×1.30 | ×0.80 |
| Market-wide | +5 pp | ×1.15 | ×0.90 |
| Combined | +5 pp | ×1.45 | ×0.75 |

In [ ]:
# Re-use the LCR inputs from notebook 05_liquidity_ratios
stress_assets = [
    HQLA_Asset("Cash + CB",      HQLA_Level.L1,            200_000_000),
    HQLA_Asset("Govt bonds",     HQLA_Level.L1,            300_000_000),
    HQLA_Asset("Covered bonds",  HQLA_Level.L2A,           120_000_000),
    HQLA_Asset("Corp bonds",     HQLA_Level.L2B_CORPORATE,  60_000_000),
]
stress_outflows = [
    Cash_Outflow("Retail stable",      1_000_000_000, Outflow_Type.RETAIL_STABLE),
    Cash_Outflow("Retail less stable",   300_000_000, Outflow_Type.RETAIL_LESS_STABLE),
    Cash_Outflow("Operational",          200_000_000, Outflow_Type.OPERATIONAL_DEPOSIT),
    Cash_Outflow("Non-fin corporate",    150_000_000, Outflow_Type.NON_FINANCIAL_CORP),
    Cash_Outflow("Financial inst",        80_000_000, Outflow_Type.FINANCIAL_INST),
]
stress_inflows = [
    Cash_Inflow("Loan repayments",  40_000_000, Inflow_Type.RETAIL),
    Cash_Inflow("Wholesale",        60_000_000, Inflow_Type.WHOLESALE),
]

base_lcr = SA_LCR_Calculator().compute(stress_assets, stress_outflows, stress_inflows)
stress_results = Liquidity_Stress_Calculator().compute(stress_assets, stress_outflows, stress_inflows)

rows = [{"scenario": "Base (unstressed)", "LCR": base_lcr.lcr,
         "HQLA_M": base_lcr.hqla/1e6, "Net_outflows_M": base_lcr.net_outflows/1e6,
         "survival_days": base_lcr.hqla/(base_lcr.net_outflows/30), "passes": base_lcr.passes}]
for sr in stress_results:
    rows.append({"scenario": sr.scenario, "LCR": sr.lcr_stressed,
                 "HQLA_M": sr.hqla_stressed/1e6, "Net_outflows_M": sr.net_outflows_stressed/1e6,
                 "survival_days": sr.survival_days, "passes": sr.passes})

df = pd.DataFrame(rows).set_index("scenario")
df["LCR_%"] = (df["LCR"] * 100).round(1)
df["survival_days"] = df["survival_days"].round(0)
df[["LCR_%", "HQLA_M", "Net_outflows_M", "survival_days", "passes"]].round(1)

In [ ]:
scenarios  = df.index.tolist()
lcr_vals   = df["LCR_%"].tolist()
survival   = df["survival_days"].tolist()
bar_colors = [p["green"] if v >= 100 else p["red"] for v in lcr_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.bar(scenarios, lcr_vals, color=bar_colors, alpha=0.85, width=0.5)
ax1.axhline(100, color=p["amber"], lw=1.5, ls="--", label="Min 100%")
ax1.set_ylabel("LCR (%)")
ax1.set_title("LCR — base vs stressed scenarios", color=p["text_title"])
ax1.tick_params(axis="x", rotation=15)
ax1.legend(fontsize=8)
ax1.grid(axis="y", alpha=0.3)

surv_colors = [p["green"] if v >= 30 else p["amber"] if v >= 15 else p["red"] for v in survival]
ax2.bar(scenarios, survival, color=surv_colors, alpha=0.85, width=0.5)
ax2.axhline(30, color=p["amber"], lw=1.2, ls=":", label="30-day LCR horizon")
ax2.set_ylabel("Survival days")
ax2.set_title("Survival period (HQLA / daily net outflows)", color=p["text_title"])
ax2.tick_params(axis="x", rotation=15)
ax2.legend(fontsize=8)
ax2.grid(axis="y", alpha=0.3)

fig.suptitle("Liquidity Stress Testing — EBA/GL/2018/02",
             color=p["text_title"], fontweight="bold")
fig.tight_layout()
plt.show()

## 6. Early Warning Indicators

Constructed directly from the metrics computed above. Thresholds represent
typical internal management buffer targets (above regulatory minimums).

In [ ]:
base_nsfr_r = SA_NSFR_Calculator().compute(
    [ASF_Item("T1", 200_000_000, category=ASF_Category.TIER1_CAPITAL),
     ASF_Item("Retail", 800_000_000, category=ASF_Category.RETAIL_STABLE_GT6M),
     ASF_Item("Corp", 200_000_000, category=ASF_Category.CORP_NON_FIN_GT6M)],
    [RSF_Item("HQLA L1", 200_000_000, category=RSF_Category.HQLA_L1_UNENCUMBERED),
     RSF_Item("Mortgages", 400_000_000, category=RSF_Category.LOANS_RETAIL_MORTGAGE_GT1Y),
     RSF_Item("Corp loans", 350_000_000, category=RSF_Category.LOANS_CORPORATE_GT1Y)],
)

indicators = [
    lcr_indicator(base_lcr.lcr),
    nsfr_indicator(base_nsfr_r.nsfr),
    encumbrance_indicator(enc.encumbrance_ratio),
    rollover_indicator(gap_result.rollover_ratio_30d),
    intraday_utilisation_indicator(intra.utilisation_rate),
]

dashboard = EWI_Monitor().evaluate(indicators)

print(f"Overall status : {dashboard.overall_status.upper()}")
print(f"Green: {dashboard.green_count}  Amber: {dashboard.amber_count}  Red: {dashboard.red_count}")
dashboard.indicators[["name", "value", "green_threshold", "amber_threshold", "status"]].round(2)

In [ ]:
STATUS_COLORS = {"green": "#38ef7d", "amber": "#f9c74f", "red": "#ff6b6b"}

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axis("off")

ind_df = dashboard.indicators
n = len(ind_df)
for i, row in ind_df.iterrows():
    color = STATUS_COLORS[row["status"]]
    ax.add_patch(mpatches.FancyBboxPatch(
        (i / n + 0.01, 0.1), 0.9 / n, 0.8,
        boxstyle="round,pad=0.02", facecolor=color, alpha=0.85,
        transform=ax.transAxes, clip_on=False,
    ))
    ax.text((i + 0.5) / n, 0.62, row["name"],
            ha="center", va="center", fontsize=8, fontweight="bold",
            transform=ax.transAxes, color="#1a1f2e")
    ax.text((i + 0.5) / n, 0.38, f"{row['value']:.1f}",
            ha="center", va="center", fontsize=10, fontweight="bold",
            transform=ax.transAxes, color="#1a1f2e")
    ax.text((i + 0.5) / n, 0.18, row["status"].upper(),
            ha="center", va="center", fontsize=7,
            transform=ax.transAxes, color="#1a1f2e")

ax.set_title(f"EWI Dashboard — Overall: {dashboard.overall_status.upper()}",
             color=STATUS_COLORS[dashboard.overall_status.value], fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. ILAAP — Internal Liquidity Adequacy Assessment

Compiles all liquidity risk results into a single ILAAP summary report.
The adequacy status reflects whether the institution meets all regulatory
minimums under both normal and stressed conditions.

In [ ]:
ilaap = ILAAP_Aggregator().compile(
    lcr_result    = base_lcr,
    nsfr_result   = base_nsfr_r,
    stress_results= stress_results,
    ewi_dashboard = dashboard,
)

print(f"Adequacy status : {ilaap.adequacy_status.upper()}")
print()
ilaap.summary

In [ ]:
STATUS_PALETTE = {"PASS": p["green"], "WARN": p["amber"], "FAIL": p["red"]}

summary = ilaap.summary.copy()
fig, ax = plt.subplots(figsize=(12, 0.55 * len(summary) + 1.2))
ax.axis("off")

cols = ["metric", "value", "threshold", "status", "detail"]
col_x = [0.0, 0.28, 0.43, 0.56, 0.65]
col_headers = ["Metric", "Value", "Threshold", "Status", "Detail"]

for cx, ch in zip(col_x, col_headers):
    ax.text(cx, 1.0, ch, ha="left", va="bottom", fontsize=9,
            fontweight="bold", color=p["text_title"], transform=ax.transAxes)

ax.axhline(0.98, color=p["border"], lw=0.8, transform=ax.transAxes)

for i, row in summary.iterrows():
    y = 1.0 - (i + 1) * (0.9 / len(summary)) - 0.05
    sc = STATUS_PALETTE.get(row["status"], p["text_muted"])
    for cx, col in zip(col_x, cols):
        text = str(row[col])
        color = sc if col == "status" else p["text_body"]
        weight = "bold" if col == "status" else "normal"
        ax.text(cx, y, text[:45], ha="left", va="center", fontsize=8,
                color=color, fontweight=weight, transform=ax.transAxes)

title_color = {"adequate": p["green"], "concerns": p["amber"], "inadequate": p["red"]}
fig.suptitle(f"ILAAP Summary — Liquidity Adequacy: {ilaap.adequacy_status.upper()}",
             color=title_color[ilaap.adequacy_status], fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()